# Predicción de MIC y fenotipo de 300 contigs plasmídicos: k-mer topológico (varios parámetros) vs k-mer clásico

Réplica ampliada del notebook de 82 genomas. Cambios principales:

- **300 genomas plasmídicos** de *Escherichia*.
- **4 configuraciones** de k-mer topológico (`k2_r2_max20`, `k2_r4_max48`, `k2_r5_max50`, `k2_r10_max100`), cada una con sus vectores **B** y **E**.
- **K-mer clásico** (vector de frecuencias, k=2) como comparación directa.
- **Modelos**: Random Forest, SVM (RBF), KNN (k=1 como en el pipeline, y k=5) y un *baseline* trivial (media / clase mayoritaria).
- **Dos tareas**: regresión del MIC y clasificación del fenotipo (*susceptible / intermediate / resistant*).
- Evaluación con **5-fold CV estratificada por fenotipo**, con los mismos folds para todos los vectores y modelos, para que la comparación sea justa.

## Librerías

In [ ]:
import warnings
from math import ceil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    accuracy_score, balanced_accuracy_score, f1_score,
    ConfusionMatrixDisplay,
)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
pd.set_option("display.width", 200)

## Configuración

In [ ]:
SEED = 42
N_GENOMES = 300          # solo para verificar que cuadre con lo esperado
CV_FOLDS = 5
LOG2_MIC = True          # True: se regresa log2(MIC) (el MIC va de ~0.015 a 64+ y crece en potencias de 2)

PROJECT = Path("/files2/gerardo/KmerTopology")
DATA_DIR = PROJECT / "data" / "processed"
MANIFEST = DATA_DIR / "selected_genomes" / "plasmid_manifest.csv"
OUT_DIR = PROJECT / "results" / "prediction_contig_300"

# (k, step_size, max_step) de cada carpeta kmer_topology_k{k}_r{step}_max{max}
TOPO_CONFIGS = [(2, 2, 20), (2, 4, 48), (2, 5, 50), (2, 10, 100)]
K_CLASSIC = 2            # carpeta kmer_k{K_CLASSIC}

### Registro de conjuntos de vectores

Cada conjunto es un par (archivo de vectores, archivo de nombres). Para el topológico hay dos matrices por configuración: **B** y **E**.

In [ ]:
FEATURE_SETS = {}

for k, r, m in TOPO_CONFIGS:
    folder = DATA_DIR / f"kmer_topology_k{k}_r{r}_max{m}"
    for tag in ["B", "E"]:
        FEATURE_SETS[f"top{tag} k{k} r{r} max{m}"] = dict(
            family=f"Topológico {tag}", r=r, m=m,
            vec=folder / f"vectors_top{tag}_escherichia.csv",
            names=folder / "escherichia_plasmid.csv",
        )

folder = DATA_DIR / f"kmer_k{K_CLASSIC}"
CLASSIC_NAME = f"kmer clásico k{K_CLASSIC}"
FEATURE_SETS[CLASSIC_NAME] = dict(
    family="K-mer clásico", r=0, m=0,
    vec=folder / "vectors_frec_escherichia.csv",
    names=folder / "escherichia_plasmid.csv",
)

for name, spec in FEATURE_SETS.items():
    print(f"{name:28s}", "OK" if spec["vec"].exists() and spec["names"].exists() else "NO ENCONTRADO")

## Datos

### Manifiesto: accesión + fenotipo + MIC

In [ ]:
id_column, label_column, mic_column = "accession", "phenotype", "mic"

df_manifest = pd.read_csv(MANIFEST, dtype=str)
df_manifest = df_manifest.dropna(subset=[id_column, label_column]).drop_duplicates(subset=id_column)

df_manifest["mic_num"] = pd.to_numeric(df_manifest[mic_column], errors="coerce")
n_sin_mic = int(df_manifest["mic_num"].isna().sum())
print(f"Accesiones con fenotipo: {len(df_manifest)} | sin MIC numérico (se excluyen): {n_sin_mic}")

df_manifest = df_manifest.dropna(subset=["mic_num"])
label_lookup = dict(zip(df_manifest[id_column], df_manifest[label_column]))
mic_lookup = dict(zip(df_manifest[id_column], df_manifest["mic_num"]))

### Carga y alineación de los vectores

Cada conjunto se indexa por accesión y se queda solo con las accesiones presentes **en todos** los conjuntos y en el manifiesto. Así todos los vectores se evalúan sobre exactamente los mismos genomas y en el mismo orden (el orden de las filas no tiene por qué coincidir entre carpetas).

In [ ]:
def load_set(vec_path, names_path):
    X = pd.read_csv(vec_path, header=None).to_numpy(dtype=float)
    names = pd.read_csv(names_path)
    col = "name" if "name" in names.columns else names.columns[0]
    ids = names[col].astype(str).tolist()
    if len(ids) != X.shape[0]:
        raise ValueError(f"{vec_path.name}: {X.shape[0]} vectores pero {len(ids)} nombres")
    df = pd.DataFrame(X, index=ids)
    return df[~df.index.duplicated(keep="first")]

raw = {}
for name, spec in FEATURE_SETS.items():
    if spec["vec"].exists() and spec["names"].exists():
        raw[name] = load_set(spec["vec"], spec["names"])
    else:
        print("Se omite (no encontrado):", name)

ids_comunes = [a for a in df_manifest[id_column]
               if all(a in df.index for df in raw.values())]

DATA = {}
for name, df in raw.items():
    X = df.loc[ids_comunes].to_numpy()
    if not np.isfinite(X).all():
        print(f"Aviso: {name} tiene NaN/inf; se reemplazan por 0")
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    DATA[name] = X

y_label = np.array([label_lookup[a] for a in ids_comunes])
y_mic = np.array([mic_lookup[a] for a in ids_comunes], dtype=float)
y_reg = np.log2(y_mic) if LOG2_MIC else y_mic

print(f"Genomas comunes a todos los conjuntos: {len(ids_comunes)} (esperados: {N_GENOMES})")
print("\nFenotipos:\n", pd.Series(y_label).value_counts().to_string())

SET_META = {n: FEATURE_SETS[n] for n in DATA}
SET_ORDER = sorted(DATA, key=lambda n: (SET_META[n]["family"], SET_META[n]["r"]))
pd.DataFrame({
    "familia": [SET_META[n]["family"] for n in SET_ORDER],
    "n_features": [DATA[n].shape[1] for n in SET_ORDER],
}, index=SET_ORDER)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(y_reg, bins=20, color="steelblue", edgecolor="white")
axes[0].set_xlabel("log2(MIC)" if LOG2_MIC else "MIC"); axes[0].set_ylabel("genomas"); axes[0].set_title("Distribución del MIC")
vc = pd.Series(y_label).value_counts()
axes[1].bar(vc.index, vc.values, color="indianred"); axes[1].set_title("Distribución del fenotipo")
plt.tight_layout(); plt.show()

## Modelos

- **Baseline**: predice la media (regresión) o la clase mayoritaria (clasificación). Sirve de piso: un vector solo aporta información si supera claramente al baseline.
- **Random Forest**: 300 árboles.
- **SVM (RBF)**: con `StandardScaler` dentro del pipeline.
- **KNN k=1** (el valor del pipeline) y **KNN k=5**: con `StandardScaler`.
- En clasificación, RF y SVM usan `class_weight="balanced"` porque *intermediate* es una clase pequeña.

El escalado va dentro del pipeline, así que en cada fold se ajusta solo con los datos de entrenamiento (sin fuga de información).

In [ ]:
def reg_models():
    return {
        "Baseline": DummyRegressor(strategy="mean"),
        "Random Forest": RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1),
        "SVM (RBF)": make_pipeline(StandardScaler(), SVR(kernel="rbf", C=1.0, gamma="scale")),
        "KNN k=1": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=1)),
        "KNN k=5": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5)),
    }

def clf_models():
    return {
        "Baseline": DummyClassifier(strategy="most_frequent"),
        "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1),
        "SVM (RBF)": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced")),
        "KNN k=1": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1)),
        "KNN k=5": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    }

MODEL_ORDER = list(reg_models())
MODELS_NO_BASE = [m for m in MODEL_ORDER if m != "Baseline"]
COLORS = dict(zip(MODEL_ORDER, ["gray", "tab:green", "tab:blue", "tab:orange", "tab:red"]))

# Mismos folds para todo (estratificados por fenotipo)
skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(np.zeros(len(y_label)), y_label))

## Validación cruzada

In [ ]:
def metrics_reg(y, p):
    rho = np.nan
    if np.std(p) > 0:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            rho = spearmanr(y, p).correlation
    return dict(R2=r2_score(y, p), RMSE=float(np.sqrt(mean_squared_error(y, p))),
                MAE=mean_absolute_error(y, p), Spearman=rho)

def metrics_clf(y, p):
    return dict(Accuracy=accuracy_score(y, p), BalAcc=balanced_accuracy_score(y, p),
                F1_macro=f1_score(y, p, average="macro", zero_division=0))

def run_cv(build_models, y, metric_fn, oof_dtype):
    rows, oof = [], {}
    for set_name in SET_ORDER:
        X = DATA[set_name]
        for model_name, est in build_models().items():
            pred = np.empty(len(y), dtype=oof_dtype)
            for fold, (tr, te) in enumerate(FOLDS):
                m = clone(est).fit(X[tr], y[tr])
                pred[te] = m.predict(X[te])
                rows.append(dict(set=set_name, model=model_name, fold=fold, **metric_fn(y[te], pred[te])))
            oof[(set_name, model_name)] = pred
    return pd.DataFrame(rows), oof

res_reg, oof_reg = run_cv(reg_models, y_reg, metrics_reg, float)
print("Regresión (MIC) lista:", res_reg.shape)

In [ ]:
res_clf, oof_clf = run_cv(clf_models, y_label, metrics_clf, object)
print("Clasificación (fenotipo) lista:", res_clf.shape)

## Resultados — Regresión del MIC

Tabla completa (media ± desviación estándar sobre los 5 folds). R² cercano a 0 o negativo significa que el modelo no mejora a predecir siempre la media.

In [ ]:
def mean_std_table(res, metrics):
    g = res.groupby(["set", "model"])[metrics].agg(["mean", "std"])
    out = pd.DataFrame(index=g.index)
    for m in metrics:
        out[m] = g[m]["mean"].map("{:.3f}".format) + " ± " + g[m]["std"].map("{:.3f}".format)
    return out.reindex(pd.MultiIndex.from_product([SET_ORDER, MODEL_ORDER], names=["set", "model"]))

def piv(res, metric, stat="mean"):
    return (res.groupby(["set", "model"])[metric].agg(stat).unstack("model")
               .reindex(index=SET_ORDER, columns=MODEL_ORDER))

display(mean_std_table(res_reg, ["R2", "RMSE", "MAE", "Spearman"]))

### Heatmaps: R² y Spearman (media de los folds)

In [ ]:
def heatmap(ax, df, title, fmt="{:.2f}", cmap="viridis", vmin=None, vmax=None):
    im = ax.imshow(df.values.astype(float), aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(df.shape[1])); ax.set_xticklabels(df.columns, rotation=30, ha="right")
    ax.set_yticks(range(df.shape[0])); ax.set_yticklabels(df.index)
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            v = df.values[i, j]
            if np.isnan(v):
                ax.text(j, i, "n/a", ha="center", va="center", color="gray", fontsize=8); continue
            ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=8,
                    color="black" if im.norm(v) > 0.55 else "white")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)

r2_mean, rho_mean = piv(res_reg, "R2"), piv(res_reg, "Spearman")
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
heatmap(axes[0], r2_mean, "R² (CV, media) — MIC", vmin=max(np.nanmin(r2_mean.values), -1.0))
heatmap(axes[1], rho_mean, "Spearman (CV, media) — MIC")
plt.tight_layout(); plt.show()

### Barras con desviación estándar entre folds

In [ ]:
def grouped_bars(res, metric, ylabel, title, models=MODEL_ORDER, hline=None):
    mean, std = piv(res, metric, "mean"), piv(res, metric, "std")
    fig, ax = plt.subplots(figsize=(14, 4.8))
    w = 0.8 / len(models); x = np.arange(len(SET_ORDER))
    for i, mdl in enumerate(models):
        ax.bar(x + i * w - 0.4 + w / 2, mean[mdl], w, yerr=std[mdl], capsize=2,
               label=mdl, color=COLORS[mdl], alpha=0.9)
    if hline is not None: ax.axhline(hline, color="k", lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels(SET_ORDER, rotation=30, ha="right")
    ax.set_ylabel(ylabel); ax.set_title(title); ax.legend(ncol=len(models), fontsize=8)
    plt.tight_layout(); plt.show()

grouped_bars(res_reg, "R2", "R² (CV)", "Regresión del MIC: R² por vector y modelo", hline=0)

### Efecto de los parámetros del k-mer topológico

Cada línea es un modelo; el eje x ordena las configuraciones por `step_size`. Las líneas punteadas horizontales son el k-mer clásico con el mismo modelo.

In [ ]:
def sweep_plot(res, metric, ylabel, title):
    mean = piv(res, metric)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
    for ax, fam in zip(axes, ["Topológico B", "Topológico E"]):
        sets = sorted([s for s in SET_ORDER if SET_META[s]["family"] == fam], key=lambda s: SET_META[s]["r"])
        xs = np.arange(len(sets))
        for mdl in MODELS_NO_BASE:
            ax.plot(xs, mean.loc[sets, mdl], marker="o", color=COLORS[mdl], label=mdl)
            if CLASSIC_NAME in mean.index:
                ax.axhline(mean.loc[CLASSIC_NAME, mdl], ls="--", color=COLORS[mdl], alpha=0.6)
        ax.axhline(mean["Baseline"].mean(), ls=":", color="gray", label="Baseline")
        ax.set_xticks(xs); ax.set_xticklabels([f"r{SET_META[s]['r']}\nmax{SET_META[s]['m']}" for s in sets])
        ax.set_title(fam); ax.set_xlabel("configuración (step_size, max_step)")
    axes[0].set_ylabel(ylabel); axes[0].legend(fontsize=8)
    fig.suptitle(title + "  (punteado: k-mer clásico)"); plt.tight_layout(); plt.show()

sweep_plot(res_reg, "R2", "R² (CV)", "R² vs parámetros topológicos")

### Valor real vs predicho (mejor modelo por vector)

In [ ]:
PHENO_COLORS = {"susceptible": "tab:green", "intermediate": "tab:orange", "resistant": "tab:red"}

best_model_reg = {s: r2_mean.loc[s, MODELS_NO_BASE].astype(float).idxmax() for s in SET_ORDER}
ncols = 3; nrows = ceil(len(SET_ORDER) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 4.2 * nrows), squeeze=False)
lims = [y_reg.min() - 0.5, y_reg.max() + 0.5]
for ax, s in zip(axes.ravel(), SET_ORDER):
    mdl = best_model_reg[s]; p = oof_reg[(s, mdl)]
    for ph, c in PHENO_COLORS.items():
        idx = y_label == ph
        ax.scatter(y_reg[idx], p[idx], s=14, alpha=0.6, color=c, label=ph)
    ax.plot(lims, lims, "k--", lw=0.8); ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_title(f"{s}\n{mdl} | R²={r2_mean.loc[s, mdl]:.2f}", fontsize=9)
    ax.set_xlabel("real"); ax.set_ylabel("predicho (CV)")
for ax in axes.ravel()[len(SET_ORDER):]: ax.axis("off")
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.show()

## Resultados — Clasificación del fenotipo

La métrica principal es la *balanced accuracy* (media del recall por clase) y el F1 macro, porque con clases desbalanceadas la accuracy sola engaña: el baseline ya obtiene la proporción de la clase mayoritaria.

In [ ]:
display(mean_std_table(res_clf, ["Accuracy", "BalAcc", "F1_macro"]))

In [ ]:
ba_mean, f1_mean = piv(res_clf, "BalAcc"), piv(res_clf, "F1_macro")
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
heatmap(axes[0], ba_mean, "Balanced accuracy (CV, media)", cmap="magma")
heatmap(axes[1], f1_mean, "F1 macro (CV, media)", cmap="magma")
plt.tight_layout(); plt.show()

In [ ]:
grouped_bars(res_clf, "BalAcc", "Balanced accuracy (CV)", "Clasificación del fenotipo: balanced accuracy por vector y modelo")
sweep_plot(res_clf, "BalAcc", "Balanced accuracy (CV)", "Balanced accuracy vs parámetros topológicos")

### Matrices de confusión (mejor combinación topológica vs mejor k-mer clásico)

In [ ]:
def best_combo(mean_df, sets):
    sub = mean_df.loc[sets, MODELS_NO_BASE].astype(float)
    s, m = sub.stack().idxmax()
    return s, m, sub.loc[s, m]

topo_sets = [s for s in SET_ORDER if SET_META[s]["family"].startswith("Topológico")]
combos = [("Mejor topológico", *best_combo(ba_mean, topo_sets))]
if CLASSIC_NAME in DATA:
    combos.append(("K-mer clásico", *best_combo(ba_mean, [CLASSIC_NAME])))

labels_order = [l for l in ["susceptible", "intermediate", "resistant"] if l in set(y_label)]
fig, axes = plt.subplots(1, len(combos), figsize=(5.8 * len(combos), 5), squeeze=False)
for ax, (tag, s, m, v) in zip(axes[0], combos):
    ConfusionMatrixDisplay.from_predictions(
        y_label, oof_clf[(s, m)], labels=labels_order, normalize="true",
        values_format=".2f", cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(f"{tag}\n{s} + {m}\nBalAcc={v:.2f}", fontsize=9)
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

## Resumen global

In [ ]:
summary = pd.DataFrame({
    "familia": [SET_META[s]["family"] for s in SET_ORDER],
    "mejor_modelo_MIC": [best_model_reg[s] for s in SET_ORDER],
    "R2": [r2_mean.loc[s, best_model_reg[s]] for s in SET_ORDER],
    "Spearman": [rho_mean.loc[s, best_model_reg[s]] for s in SET_ORDER],
    "mejor_modelo_fenotipo": [ba_mean.loc[s, MODELS_NO_BASE].astype(float).idxmax() for s in SET_ORDER],
    "BalAcc": [ba_mean.loc[s, MODELS_NO_BASE].astype(float).max() for s in SET_ORDER],
    "F1_macro": [f1_mean.loc[s, ba_mean.loc[s, MODELS_NO_BASE].astype(float).idxmax()] for s in SET_ORDER],
}, index=SET_ORDER)

base_r2 = r2_mean["Baseline"].mean(); base_ba = ba_mean["Baseline"].mean()
print(f"Baseline: R² ≈ {base_r2:.3f} | balanced accuracy ≈ {base_ba:.3f}")
display(summary.sort_values("R2", ascending=False).style.format(
    {"R2": "{:.3f}", "Spearman": "{:.3f}", "BalAcc": "{:.3f}", "F1_macro": "{:.3f}"}
).background_gradient(subset=["R2", "BalAcc"], cmap="YlGn"))

In [ ]:
mejor_mic = summary["R2"].idxmax(); mejor_fen = summary["BalAcc"].idxmax()
print(f"Mejor para MIC:      {mejor_mic} + {summary.loc[mejor_mic, 'mejor_modelo_MIC']}  (R²={summary.loc[mejor_mic, 'R2']:.3f})")
print(f"Mejor para fenotipo: {mejor_fen} + {summary.loc[mejor_fen, 'mejor_modelo_fenotipo']}  (BalAcc={summary.loc[mejor_fen, 'BalAcc']:.3f})")
if CLASSIC_NAME in summary.index:
    t = summary.loc[topo_sets]
    print(f"\nMejor topológico vs k-mer clásico:")
    print(f"  R²     {t['R2'].max():.3f}  vs  {summary.loc[CLASSIC_NAME, 'R2']:.3f}")
    print(f"  BalAcc {t['BalAcc'].max():.3f}  vs  {summary.loc[CLASSIC_NAME, 'BalAcc']:.3f}")
print("\nNota: compara siempre contra el baseline y contra la desviación entre folds; diferencias menores que la std no son concluyentes.")

## Guardar resultados

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
res_reg.to_csv(OUT_DIR / "cv_regresion_mic_por_fold.csv", index=False)
res_clf.to_csv(OUT_DIR / "cv_clasificacion_fenotipo_por_fold.csv", index=False)
summary.to_csv(OUT_DIR / "resumen_mejor_por_vector.csv")
print("Guardado en", OUT_DIR)